# TrackLink USBL — Parsed Fixes Visualisation

Loads a batch of parsed TrackLink fix files and, for each deployment, plots the
ship track and target positions derived from the 2D range-bearing model
(absolute and relative bearing) on an interactive map, plus time series of
bearing, slant range, and target XYZ offsets.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
LABELS: list[str] = [
    "qdch0ftq_20100428_020202",
    "qdch0ftq_20110415_020103",
    "qdch0ftq_20120430_002423",
    "qdch0ftq_20130406_023610",
    "qdch0ftq_20140327_071251",
]

# qd61g27j_20100421_022145
# qd61g27j_20110410_011202
# qd61g27j_20120422_043114
# qd61g27j_20130414_013620

# qdc5ghs3_20100430_024508
# qdc5ghs3_20120501_033336
# qdc5ghs3_20130405_103429

# qdch0ftq_20100428_020202
# qdch0ftq_20110415_020103
# qdch0ftq_20120430_002423
# qdch0ftq_20130406_023610
# qdch0ftq_20140327_071251

# qdchdmy1_20110416_005411
# qdchdmy1_20130406_081713
# qdchdmy1_20140328_063358

# qtqxshxs_20110815_102540
# qtqxshxs_20150327_015552
# qtqxshxs_20150328_000850
# qtqxshxs_20150328_042551

# r234xgje_20100604_230524
# r234xgje_20120530_064545
# r234xgje_20140616_205232

# r23685bc_20100605_021022
# r23685bc_20120530_233021
# r23685bc_20140616_225022

# r23m7ms0_20100606_001908
# r23m7ms0_20120601_070118
# r23m7ms0_20140616_044549

# r29mrd12_20090613_010853
# r29mrd12_20090613_104954
# r29mrd12_20110612_045149
# r29mrd12_20130611_015335

# r29mrd5h_20090612_225306
# r29mrd5h_20090613_100254
# r29mrd5h_20110612_033752
# r29mrd5h_20130611_002419

# r7jjskxq_20101023_210332
# r7jjskxq_20121013_060425
# r7jjskxq_20131022_004934

# r7jjss8n_20101023_210332
# r7jjss8n_20121013_060425
# r7jjss8n_20131022_004934

# r7jjssbh_20101023_210332
# r7jjssbh_20121013_060425
# r7jjssbh_20131022_004934

FIXES_DIR: Path = Path("/data/exos_01/acfr_tracklink_logs_v2_parsed")

## Load data

In [ ]:
REQUIRED_COLS: list[str] = [
    "timestamp",
    "ship_latitude",
    "ship_longitude",
    "ship_heading",
    "ship_roll",
    "ship_pitch",
    "target_bearing_angle",
    "target_slant_range",
]

fixes_by_label: dict[str, pd.DataFrame] = {}

for label in LABELS:
    path: Path = FIXES_DIR / f"{label}_tracklink_fixes.csv"
    if not path.exists():
        print(f"[skip] {label} — file not found")
        continue

    fixes: pd.DataFrame = pd.read_csv(
        path, parse_dates=["timestamp"], date_format="ISO8601"
    )

    missing: list[str] = [
        col for col in REQUIRED_COLS if col not in fixes.columns
    ]
    if missing:
        print(f"[skip] {label} — missing columns: {missing}")
        continue

    fixes_by_label[label] = fixes
    has_xyz: bool = all(
        col in fixes.columns for col in ("target_x", "target_y", "target_z")
    ) and fixes[["target_x", "target_y", "target_z"]].notna().any().all()
    print(
        f"[ok]   {label} — {len(fixes)} rows"
        f"  XYZ: {'present' if has_xyz else 'absent'}"
    )

print(f"\nLoaded {len(fixes_by_label)} / {len(LABELS)} deployments")

## Derive target lat/lon — 2D simplified bearing-range models

Two flat-earth approximations that project the slant range onto the horizontal
plane and use `pymap3d.ned2geodetic` to convert N/E offsets to geographic
coordinates. Results are stored as `target_lat_2d_*` / `target_lon_2d_*`
columns to distinguish them from the XYZ-derived positions computed later.

- **Absolute bearing** (`_2d_absolute`): bearing is a compass azimuth (° from
  North, CW).
- **Relative bearing** (`_2d_relative`): bearing is relative to the ship's
  heading; the heading is added before converting to N/E components.

In [ ]:
import pymap3d


def derive_target_latlon_absolute_bearing(
    ship_lat: np.ndarray,
    ship_lon: np.ndarray,
    bearing_deg: np.ndarray,
    slant_range_m: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Derive target lat/lon assuming bearing is an absolute compass azimuth.

    Arguments
    ---------
    ship_lat: Ship latitude in decimal degrees.
    ship_lon: Ship longitude in decimal degrees.
    bearing_deg: Absolute bearing to target in degrees from North, clockwise.
    slant_range_m: Slant range to target in metres (treated as horizontal
        distance).

    Returns
    -------
    Tuple of (target_lat, target_lon) arrays in decimal degrees.
    """
    bearing_rad = np.radians(bearing_deg)
    north_m = slant_range_m * np.cos(bearing_rad)
    east_m = slant_range_m * np.sin(bearing_rad)
    target_lat, target_lon, _ = pymap3d.ned2geodetic(
        north_m,
        east_m,
        np.zeros_like(north_m),
        ship_lat,
        ship_lon,
        np.zeros_like(north_m),
    )
    return target_lat, target_lon


def derive_target_latlon_relative_bearing(
    ship_lat: np.ndarray,
    ship_lon: np.ndarray,
    ship_heading_deg: np.ndarray,
    bearing_deg: np.ndarray,
    slant_range_m: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Derive target lat/lon assuming bearing is relative to ship heading.

    Arguments
    ---------
    ship_lat: Ship latitude in decimal degrees.
    ship_lon: Ship longitude in decimal degrees.
    ship_heading_deg: Ship heading in degrees from North, clockwise.
    bearing_deg: Bearing to target relative to ship heading, in degrees.
    slant_range_m: Slant range to target in metres (treated as horizontal
        distance).

    Returns
    -------
    Tuple of (target_lat, target_lon) arrays in decimal degrees.
    """
    absolute_bearing_deg = (ship_heading_deg + bearing_deg) % 360
    return derive_target_latlon_absolute_bearing(
        ship_lat,
        ship_lon,
        absolute_bearing_deg,
        slant_range_m,
    )


for label, fixes in fixes_by_label.items():
    fixes["target_lat_2d_absolute"], fixes["target_lon_2d_absolute"] = (
        derive_target_latlon_absolute_bearing(
            fixes["ship_latitude"].to_numpy(),
            fixes["ship_longitude"].to_numpy(),
            fixes["target_bearing_angle"].to_numpy(),
            fixes["target_slant_range"].to_numpy(),
        )
    )
    fixes["target_lat_2d_relative"], fixes["target_lon_2d_relative"] = (
        derive_target_latlon_relative_bearing(
            fixes["ship_latitude"].to_numpy(),
            fixes["ship_longitude"].to_numpy(),
            fixes["ship_heading"].to_numpy(),
            fixes["target_bearing_angle"].to_numpy(),
            fixes["target_slant_range"].to_numpy(),
        )
    )

## Overview map: ship track and derived target positions

In [ ]:
for label, fixes in fixes_by_label.items():
    t_s: np.ndarray = (fixes["timestamp"].astype(np.int64) / 1e9).to_numpy()
    elapsed: np.ndarray = (t_s - t_s.min()) / 60.0

    center_lat: float = float(fixes["ship_latitude"].mean())
    center_lon: float = float(fixes["ship_longitude"].mean())

    fig = go.Figure()

    fig.add_trace(
        go.Scattermapbox(
            lat=fixes["ship_latitude"],
            lon=fixes["ship_longitude"],
            mode="lines+markers",
            marker=dict(
                size=4,
                color=elapsed,
                colorscale="Viridis",
                cmin=0,
                cmax=float(elapsed.max()),
                showscale=True,
                colorbar=dict(title="Elapsed (min)", x=0.92),
            ),
            line=dict(width=1, color="royalblue"),
            name="Ship track",
            hovertemplate=(
                "<b>Ship</b><br>"
                "Lat: %{lat:.6f}<br>"
                "Lon: %{lon:.6f}<br>"
                "<extra></extra>"
            ),
        )
    )

    fig.add_trace(
        go.Scattermapbox(
            lat=fixes["target_lat_2d_absolute"],
            lon=fixes["target_lon_2d_absolute"],
            mode="markers",
            marker=dict(
                size=6,
                color=elapsed,
                colorscale=[[0, "crimson"], [1, "lightsalmon"]],
                cmin=0,
                cmax=float(elapsed.max()),
                showscale=False,
            ),
            name="Target — 2D absolute bearing",
            customdata=np.stack(
                [fixes["target_bearing_angle"], fixes["target_slant_range"]],
                axis=1,
            ),
            hovertemplate=(
                "<b>Target (2D absolute bearing)</b><br>"
                "Lat: %{lat:.6f}<br>"
                "Lon: %{lon:.6f}<br>"
                "Bearing: %{customdata[0]:.1f}°<br>"
                "Slant range: %{customdata[1]:.1f} m<br>"
                "<extra></extra>"
            ),
        )
    )

    fig.add_trace(
        go.Scattermapbox(
            lat=fixes["target_lat_2d_relative"],
            lon=fixes["target_lon_2d_relative"],
            mode="markers",
            marker=dict(
                size=6,
                color=elapsed,
                colorscale=[[0, "darkgreen"], [1, "lightgreen"]],
                cmin=0,
                cmax=float(elapsed.max()),
                showscale=False,
            ),
            name="Target — 2D relative bearing",
            customdata=np.stack(
                [
                    fixes["target_bearing_angle"],
                    fixes["ship_heading"],
                    fixes["target_slant_range"],
                ],
                axis=1,
            ),
            hovertemplate=(
                "<b>Target (2D relative bearing)</b><br>"
                "Lat: %{lat:.6f}<br>"
                "Lon: %{lon:.6f}<br>"
                "Bearing: %{customdata[0]:.1f}°<br>"
                "Ship heading: %{customdata[1]:.1f}°<br>"
                "Slant range: %{customdata[2]:.1f} m<br>"
                "<extra></extra>"
            ),
        )
    )

    fig.update_layout(
        title=f"Ship track and derived target positions — {label}",
        mapbox=dict(
            style="open-street-map",
            center=dict(lat=center_lat, lon=center_lon),
            zoom=14,
        ),
        legend=dict(x=0.01, y=0.99),
        height=700,
        margin=dict(l=0, r=0, t=40, b=0),
    )

    fig.show()

## Time series: target XYZ offsets (vessel frame)

Only rendered when `target_x/y/z` are present and non-NaN in the parsed CSV.
Vessel frame convention: X = forward, Y = starboard, Z = down.

In [ ]:
for label, fixes in fixes_by_label.items():
    has_xyz: bool = all(
        col in fixes.columns for col in ("target_x", "target_y", "target_z")
    ) and fixes[["target_x", "target_y", "target_z"]].notna().any().all()

    if not has_xyz:
        print(f"{label} — target_x/y/z absent or all-NaN, skipping.")
        continue

    fig_xyz = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        subplot_titles=(
            "Target X — vessel frame forward (m)",
            "Target Y — vessel frame starboard (m)",
            "Target Z — vessel frame down (m)",
        ),
        vertical_spacing=0.07,
    )

    for row, (column, color) in enumerate(
        [
            ("target_x", "steelblue"),
            ("target_y", "darkorange"),
            ("target_z", "seagreen"),
        ],
        start=1,
    ):
        fig_xyz.add_trace(
            go.Scatter(
                x=fixes["timestamp"],
                y=fixes[column],
                mode="lines+markers",
                marker=dict(size=4),
                name=column,
                line=dict(color=color),
            ),
            row=row,
            col=1,
        )
        fig_xyz.update_yaxes(title_text="m", row=row, col=1)

    fig_xyz.update_layout(
        title=f"Target XYZ offsets (vessel frame) — {label}",
        height=700,
        showlegend=False,
    )

    fig_xyz.show()